In [ ]:
from abc import ABC, abstractmethod

class Producto(ABC):
    def __init__(self, nombre, precio_base):
        self._nombre = nombre
        self._precio_base = precio_base

    @property
    def nombre(self):
        return self._nombre

    @property
    def precio_base(self):
        return self._precio_base

    @abstractmethod
    def calcular_precio_total(self):
        pass

    @abstractmethod
    def obtener_descripcion(self):
        pass

class Cliente(ABC):
    def __init__(self, nombre, email):
        self._nombre = nombre
        self._email = email

    @property
    def nombre(self):
        return self._nombre

    @property
    def email(self):
        return self._email

    @abstractmethod
    def calcular_descuento(self, subtotal):
        pass

    @abstractmethod
    def obtener_clasificacion(self):
        pass

class TamanoBebida:
    def __init__(self, nombre, costo_adicional):
        self.nombre = nombre
        self.costo_adicional = costo_adicional

class LecheEspecial:
    def __init__(self, nombre, costo_adicional):
        self.nombre = nombre
        self.costo_adicional = costo_adicional

class BebidaPreparada(Producto):
    def __init__(self, nombre, precio_base, tamano, leche = None):
        super().__init__(nombre, precio_base)
        self.tamano = tamano
        self.leche = leche

    def calcular_precio_total(self):
        total = self.precio_base + self.tamano.costo_adicional
        if self.leche:
            total += self.leche.costo_adicional
        return round(total, 2)

    def obtener_descripcion(self):
        desc = f"Bebida: {self.nombre} [{self.tamano.nombre}]"
        if self.leche:
            desc += f" (Con leche de {self.leche.nombre})"
        return desc

class AlimentoPasteleria(Producto):
    def __init__(self, nombre, precio_base, impuesto_especifico, requiere_calentado = False):
        super().__init__(nombre, precio_base)
        self.impuesto_especifico = impuesto_especifico
        self.requiere_calentado = requiere_calentado
        self._calentado_aplicado = False

    def calentar(self):
        if self.requiere_calentado:
            self._calentado_aplicado = True

    def calcular_precio_total(self):
        precio_con_impuesto = self.precio_base * (1 + self.impuesto_especifico)
        return round(precio_con_impuesto, 2)

    def obtener_descripcion(self):
        estado_calor = " [Calentado en horno]" if self._calentado_aplicado else " [Temperatura ambiente]"
        return f"Pastelería: {self.nombre}{estado_calor}"

class ComboPromocional(Producto):
    def __init__(self, nombre, descuento_porcentaje = 0.15):
        super().__init__(nombre, 0.0)
        self.productos = []
        self.descuento_porcentaje = descuento_porcentaje

    def agregar_producto(self, producto):
        self.productos.append(producto)

    def calcular_precio_total(self):
        suma_total = sum(p.calcular_precio_total() for p in self.productos)
        precio_con_descuento = suma_total * (1 - self.descuento_porcentaje)
        return round(precio_con_descuento, 2)

    def obtener_descripcion(self):
        items = " + ".join([p.nombre for p in self.productos])
        return f"Combo Promo '{self.nombre}' (Incluye: {items}) con {int(self.descuento_porcentaje * 100)}% de dcto."

class ClienteRegular(Cliente):
    def calcular_descuento(self, subtotal):
        return 0.0

    def obtener_clasificacion(self):
        return "Regular"

class ClienteCorporativo(Cliente):
    def __init__(self, nombre, email, empresa, descuento_pct = 0.10):
        super().__init__(nombre, email)
        self.empresa = empresa
        self.descuento_pct = descuento_pct

    def calcular_descuento(self, subtotal):
        return round(subtotal * self.descuento_pct, 2)

    def obtener_clasificacion(self):
        return f"Corporativo - {self.empresa} ({int(self.descuento_pct * 100)}% dcto.)"

class ClienteVIP(Cliente):
    def __init__(self, nombre, email, descuento_pct = 0.20, puntos_fidelidad = 500):
        super().__init__(nombre, email)
        self.descuento_pct = descuento_pct
        self.puntos_fidelidad = puntos_fidelidad

    def calcular_descuento(self, subtotal):
        return round(subtotal * self.descuento_pct, 2)

    def obtener_clasificacion(self):
        return f"VIP Exclusivo ({int(self.descuento_pct * 100)}% dcto. + {self.puntos_fidelidad} pts)"

class Pedido:
    _contador_id = 1001

    def __init__(self, cliente):
        self.id_pedido = Pedido._contador_id
        Pedido._contador_id += 1
        self.cliente = cliente
        self.productos = []

    def agregar_item(self, producto):
        self.productos.append(producto)

    def obtener_subtotal(self):
        return sum(p.calcular_precio_total() for p in self.productos)

    def obtener_descuento(self):
        return self.cliente.calcular_descuento(self.obtener_subtotal())

    def obtener_total_final(self):
        return round(self.obtener_subtotal() - self.obtener_descuento(), 2)

    def generar_ticket_transparente(self):
        subtotal_bruto = self.obtener_subtotal()
        descuento_cliente = self.obtener_descuento()
        total_final = self.obtener_total_final()

        lineas = []
        lineas.append("=" * 65)
        lineas.append(f"         CAFETERÍA CENTRAL - TICKET DE PEDIDO #{self.id_pedido}")
        lineas.append("=" * 65)
        lineas.append(f"Cliente      : {self.cliente.nombre} | Clasificación: [{self.cliente.obtener_clasificacion()}]")
        lineas.append("-" * 65)
        lineas.append("LÍNEAS DE PRODUCTOS Y DETALLES DE PREPARACIÓN:")
        
        for idx, prod in enumerate(self.productos, 1):
            lineas.append(f"  {idx}. {prod.obtener_descripcion()}")
            lineas.append(f"     -> Subtotal Línea: ${prod.calcular_precio_total():.2f}")
        
        lineas.append("-" * 65)
        lineas.append(f"Subtotal Consolidado de Líneas : ${subtotal_bruto:.2f}")
        lineas.append(f"Descuento por Membresía      : -${descuento_cliente:.2f}")
        lineas.append(f"TOTAL FINAL A PAGAR            : ${total_final:.2f}")
        lineas.append("=" * 65)
        
        return "\n".join(lineas)

class TurnoCafeteria:
    def __init__(self, cajero_responsable):
        self.cajero_responsable = cajero_responsable
        self.pedidos_realizados = []
        self.turno_cerrado = False

    def registrar_pedido(self, pedido):
        if self.turno_cerrado:
            raise Exception("Acción denegada: El turno ya se encuentra cerrado.")
        self.pedidos_realizados.append(pedido)
        print(f"[Sistema Turno] Pedido #{pedido.id_pedido} registrado exitosamente.")

    def calcular_recaudacion_total(self):
        return sum(p.obtener_total_final() for p in self.pedidos_realizados)

    def generar_reporte_cierre(self):
        self.turno_cerrado = True
        total_recaudado = self.calcular_recaudacion_total()
        cantidad_pedidos = len(self.pedidos_realizados)

        reporte = []
        reporte.append("=" * 70)
        reporte.append(f"          CIERRE GLOBAL DE TURNO - CAFETERÍA CENTRAL")
        reporte.append("=" * 70)
        reporte.append(f"Cajero Responsable : {self.cajero_responsable}")
        reporte.append(f"Total de Pedidos   : {cantidad_pedidos}")
        reporte.append(f"RECAUDACIÓN NETA   : ${total_recaudado:.2f}")
        reporte.append("-" * 70)
        reporte.append("RESUMEN DE TICKETS EMITIDOS:")
        for p in self.pedidos_realizados:
            reporte.append(f" -> Pedido #{p.id_pedido} | Cliente: {p.cliente.nombre} | Total: ${p.obtener_total_final():.2f}")
        reporte.append("=" * 70)

        return "\n".join(reporte)

if __name__ == "__main__":
    print("=== INICIANDO SISTEMA DE CAFETERÍA CENTRAL ===\n")

    turno = TurnoCafeteria(cajero_responsable="Lucía Méndez")

    tamano_pequeno = TamanoBebida("Pequeño (8oz)", 0.40)
    tamano_mediano = TamanoBebida("Mediano (12oz)", 0.80)
    tamano_grande = TamanoBebida("Grande (16oz)", 1.50)
    leche_almendras = LecheEspecial("Almendras", 0.90)

    cliente_regular = ClienteRegular("Catalina Gonzáles", "catalina.gonzales@gmail.com")
    cliente_corporativo = ClienteCorporativo("Jensen Huang", "jensen.huang@techcorp.com", "TechCorp", 0.15)
    cliente_vip = ClienteVIP("Sergio Pérez", "sergio.perez@gmail.com", 0.25, 1250)

    pedido1 = Pedido(cliente_regular)
    pedido1.agregar_item(BebidaPreparada("Café Latte", 3.00, tamano_grande, leche_almendras))
    croissant = AlimentoPasteleria("Croissant Mantequilla", 2.50, impuesto_especifico=0.08, requiere_calentado=True)
    croissant.calentar()
    pedido1.agregar_item(croissant)
    
    turno.registrar_pedido(pedido1)
    print("\n" + pedido1.generar_ticket_transparente() + "\n")

    pedido2 = Pedido(cliente_corporativo)
    combo = ComboPromocional("Desayuno Ejecutivo", descuento_porcentaje=0.20)
    combo.agregar_producto(BebidaPreparada("Cappuccino", 2.50, tamano_mediano))
    combo.agregar_producto(AlimentoPasteleria("Muffin Arándanos", 2.20, 0.08, False))
    pedido2.agregar_item(combo)
    
    turno.registrar_pedido(pedido2)
    print("\n" + pedido2.generar_ticket_transparente() + "\n")

    pedido3 = Pedido(cliente_vip)
    pedido3.agregar_item(BebidaPreparada("Espresso Doble", 2.00, tamano_mediano))

    turno.registrar_pedido(pedido3)
    print("\n" + pedido3.generar_ticket_transparente() + "\n")

    print("\n" + turno.generar_reporte_cierre())

=== INICIANDO SISTEMA DE CAFETERÍA CENTRAL ===

[Sistema Turno] Pedido #1001 registrado exitosamente.

         CAFETERÍA CENTRAL - TICKET DE PEDIDO #1001
Cliente      : Catalina Gonzáles | Clasificación: [Regular]
-----------------------------------------------------------------
LÍNEAS DE PRODUCTOS Y DETALLES DE PREPARACIÓN:
  1. Bebida: Café Latte [Grande (16oz)] (Con leche de Almendras)
     -> Subtotal Línea: $5.40
  2. Pastelería: Croissant Mantequilla [Calentado en horno]
     -> Subtotal Línea: $2.70
-----------------------------------------------------------------
Subtotal Consolidado de Líneas : $8.10
Descuento por Membresía      : -$0.00
TOTAL FINAL A PAGAR            : $8.10

[Sistema Turno] Pedido #1002 registrado exitosamente.

         CAFETERÍA CENTRAL - TICKET DE PEDIDO #1002
Cliente      : Carlos Gómez | Clasificación: [Corporativo - TechCorp (15% dcto.)]
-----------------------------------------------------------------
LÍNEAS DE PRODUCTOS Y DETALLES DE PREPARACIÓN:
  